# CSoT'26 - ML in Astronomy - Week 3 . Part 2: Evaluation and Interpretation (Solution)

Reference implementation. **Only open after attempting [`week3_eval_starter.ipynb`](week3_eval_starter.ipynb).**

Companion reading: [`05-evaluation-and-overfitting.md`](../05-evaluation-and-overfitting.md), [`06-confusion-matrix-and-metrics.md`](../06-confusion-matrix-and-metrics.md), [`07-saving-and-loading-models.md`](../07-saving-and-loading-models.md).

## Step 0 - Data + model

Reproduce the Week-1 pipeline and the Part-1 `GalaxyCNN`. The data-build cells are commented out (run them once on a fresh Colab, exactly as in Part 1).

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

In [ ]:
# --- Rebuild the galaxy_data/{train,val,test}/<class>/ layout if needed ---
# See week3_cnn_solution.ipynb Step 0 (or week1_data_solution.ipynb) for the
# download + build_split_imagefolder_layout(...) calls. Run those once, then:
DATA_ROOT = Path("galaxy_data")

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_ds.classes)
print("classes:", train_ds.classes, "| num_classes:", num_classes)

In [ ]:
class GalaxyCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 16 * 16, 128), nn.ReLU(), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = GalaxyCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Optionally load Part-1 weights instead of re-training:
# model.load_state_dict(torch.load('galaxy_model.pth', map_location=device))
print(model)

## Step 1 - The evaluation function

Uses `model.eval()` (inference behaviour) and `torch.no_grad()` (no gradient bookkeeping). Returns average loss and accuracy on any loader.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return total_loss / total, correct / total

## Step 2 - Train while tracking validation

In [ ]:
num_epochs = 8
train_losses, val_losses, val_accs = [], [], []

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running += loss.item() * inputs.size(0)
    train_loss = running / len(train_loader.dataset)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    train_losses.append(train_loss); val_losses.append(val_loss); val_accs.append(val_acc)
    print(f"Epoch {epoch+1:2d}  train {train_loss:.3f}  val {val_loss:.3f}  val_acc {val_acc:.3f}")

## Step 3 - Plot train vs validation loss

If the two lines track together and flatten, training is healthy. If `train` keeps falling while `val` turns upward, the widening gap is overfitting - the best model is at the lowest `val` point.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o", label="train")
plt.plot(range(1, num_epochs + 1), val_losses, marker="s", label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Train vs validation loss")
plt.grid(True, alpha=0.3)
plt.show()

## Step 4 - Final test accuracy vs the Week-2 baseline

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"FINAL test accuracy: {test_acc:.3f}")

week2_baseline = 0.55  # <-- replace with YOUR Week-2 KNN/LogReg test accuracy
print(f"Week-2 baseline: {week2_baseline:.3f}")
print(f"Improvement over baseline: {test_acc - week2_baseline:+.3f}")

## Step 5 - Confusion matrix

Collect every prediction and true label across the test set, move them to CPU, and display a labelled matrix. Rows = true class, columns = predicted; the diagonal is correct.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for inputs, targets in loader:
        preds = model(inputs.to(device)).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(targets)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

y_pred, y_true = collect_predictions(model, test_loader, device)
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_ds.classes)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Galaxy CNN - confusion matrix (test set)")
plt.show()

## Step 6 - Per-class report + astrophysical reading

In [ ]:
print(classification_report(y_true, y_pred, target_names=test_ds.classes))

**Reading the matrix (example).** The heaviest off-diagonal cells are typically `spiral` <-> `spiral_barred`: the bar is a subtle feature that can be faint or foreshortened (edge-on), so the two classes genuinely overlap - see density waves and bars in [`04`](../04-spiral-structure-and-star-formation.md). Smooth, bulge-dominated spirals also shade toward `elliptical`, the lenticular ambiguity from [`08`](../08-lenticulars-mergers-and-evolution.md). Much of this confusion mirrors where human classifiers also disagree, so it is partly a real physical continuum, not only a model failure.

## Step 7 - Save and reload (round-trip check)

Save the `state_dict`, rebuild a fresh `GalaxyCNN`, load the weights, and confirm identical test accuracy.

In [ ]:
torch.save(model.state_dict(), "galaxy_model.pth")
print("saved galaxy_model.pth")

loaded = GalaxyCNN(num_classes=num_classes).to(device)
loaded.load_state_dict(torch.load("galaxy_model.pth", map_location=device))
loaded.eval()

_, reloaded_acc = evaluate(loaded, test_loader, criterion, device)
print(f"original test acc: {test_acc:.4f}")
print(f"reloaded test acc: {reloaded_acc:.4f}")
assert abs(reloaded_acc - test_acc) < 1e-6, "Reloaded model differs - check architecture/state_dict!"
print("Round-trip verified: weights saved and restored correctly.")

## Step 8 (stretch) - Look at what the model got wrong

In [ ]:
# Plot a few misclassified test galaxies, titled 'true -> predicted'.
model.eval()
shown = 0
plt.figure(figsize=(10, 4))
with torch.no_grad():
    for inputs, targets in test_loader:
        preds = model(inputs.to(device)).argmax(dim=1).cpu()
        for img, t, p in zip(inputs, targets, preds):
            if t != p and shown < 8:
                shown += 1
                ax = plt.subplot(2, 4, shown)
                ax.imshow((img * 0.5 + 0.5).permute(1, 2, 0).numpy())
                ax.set_title(f"{test_ds.classes[t]} -> {test_ds.classes[p]}", fontsize=8)
                ax.axis("off")
        if shown >= 8:
            break
plt.suptitle("Misclassified test galaxies")
plt.show()

## Reflection (example answers)

1. **Accuracy vs baseline.** The CNN typically lands well above the Week-2 flatten-and-classify baseline. The gap exists because convolutions preserve the spatial structure (arms, bars, bulges) that flattening destroyed - the model can actually *see* morphology.
2. **Overfitting.** If validation loss bottomed out and then rose while training loss kept falling, that is overfitting; the fixes worth trying are data augmentation (flips/rotations - natural for galaxies), more data (raise `PER_CLASS`), early stopping, or dropout.
3. **Most-confused pair.** Usually `spiral` vs `spiral_barred`. It is both a model limitation (64x64 is coarse for a faint bar) and a real ambiguity (faint/edge-on bars genuinely resemble unbarred spirals).
4. **Round-trip check.** Saving can silently go wrong (wrong path, saved before training finished, architecture mismatch). Asserting that the reloaded model reproduces the exact accuracy turns 'I think I saved it' into 'I know I can ship it'.